In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║       APSSDC Internship - Attendance Analysis & ML Predictor    ║
║       Works on Zoom meeting export CSVs                         ║
╚══════════════════════════════════════════════════════════════════╝

What this script does:
  1. Loads one or more Zoom meeting-detail CSV files
  2. Cleans & deduplicates student records by email
  3. Engineers features: time attended, join frequency, engagement score
  4. Builds an overall attendance summary per student
  5. Assigns Certificate Status based on configurable threshold
  6. Trains 3 ML models to predict certificate eligibility
  7. Outputs a clean CSV report

Usage:
    python attendance_analysis.py
"""

import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score

# ──────────────────────────────────────────────
# CONFIGURATION  ← change these paths to your files
# ──────────────────────────────────────────────
CSV_FILES = [
    "/content/meetinglistdetails_2026_05_30_2026_05_30.csv",
    "/content/meetinglistdetails_2026_05_29_2026_05_29.csv",
    # Add more days here:
    # "meetinglistdetails_2026_05_28_2026_05_28.csv",
]

# A student is counted as "attended" a session if they stayed for
# at least this fraction of the session's total duration.
ATTENDANCE_THRESHOLD_PCT = 0.50      # 50% of session duration

# Minimum attendance % across ALL sessions to get a certificate
CERTIFICATE_THRESHOLD = 75.0         # >= 75% → Issued

# Short display names for long topic strings (add your own topics here)
TOPIC_LABELS = {
    "Machine Learning -  Summer Online Internship - 2026 by APSSDC": "Machine Learning",
    "Embedded Systems - Summer Online Internship - 2026 by APSSDC":  "Embedded Systems",
    "Data Analysis using Python - Summer Online Internship - 2026 by APSSDC": "Data Analysis (Python)",
    "Cloud Computing - DevOps - Summer Online Internship - 2026 by APSSDC": "Cloud & DevOps",
}

OUTPUT_CSV = "attendance_report.csv"


# ════════════════════════════════════════════════════════════════
# STEP 1 — LOAD DATA
# ════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 1 — Loading & Merging CSV Files")
print("=" * 65)

frames = []
for path in CSV_FILES:
    if os.path.exists(path):
        frames.append(pd.read_csv(path))
        print(f"  Loaded: {os.path.basename(path)}")
    else:
        print(f"  NOT FOUND: {path}")

if not frames:
    raise FileNotFoundError("No CSV files found. Check the CSV_FILES paths above.")

df = pd.concat(frames, ignore_index=True)
print(f"\n  Total rows loaded : {len(df):,}")
print(f"  Unique meeting IDs: {df['ID'].nunique()}")


# ════════════════════════════════════════════════════════════════
# STEP 2 — SESSION META TABLE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 2 — Session Information")
print("=" * 65)

sessions = (
    df.drop_duplicates(subset="ID")
    [["ID", "Topic", "Duration (minutes)", "Start time"]]
    .rename(columns={"Duration (minutes)": "Session_Duration"})
    .copy()
)
sessions["Topic_Short"] = (
    sessions["Topic"].map(TOPIC_LABELS).fillna(sessions["Topic"].str[:40])
)
sessions["Date"] = pd.to_datetime(sessions["Start time"]).dt.date

for _, row in sessions.iterrows():
    print(f"  [{row['Date']}] {row['Topic_Short']:<30} — {row['Session_Duration']} min")

TOTAL_SESSIONS = len(sessions)
print(f"\n  Total classes (sessions) in dataset: {TOTAL_SESSIONS}")


# ════════════════════════════════════════════════════════════════
# STEP 3 — CLEAN STUDENT RECORDS
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 3 — Cleaning Student Records")
print("=" * 65)

students = df[df["Guest"] == "Yes"].copy()
students["Duration (minutes).1"] = pd.to_numeric(
    students["Duration (minutes).1"], errors="coerce"
).fillna(0)

# Normalise email: lowercase + strip spaces
students["Email"] = students["Email"].str.lower().str.strip()

# Drop rows with no valid email
students = students[students["Email"].str.contains("@", na=False)]

print(f"  Student rows (after filter) : {len(students):,}")
print(f"  Unique student emails       : {students['Email'].nunique():,}")


# ════════════════════════════════════════════════════════════════
# STEP 4 — PER-SESSION ATTENDANCE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 4 — Computing Per-Session Attendance")
print("=" * 65)

# Some students rejoin mid-session — sum all their time per session
per_session = (
    students.groupby(["Email", "ID"])
    .agg(
        Name=("Name (original name)", "first"),
        Total_Time_Min=("Duration (minutes).1", "sum"),
        Join_Count=("Join time", "count"),        # number of times they joined
    )
    .reset_index()
)

# Bring in session duration
per_session = per_session.merge(
    sessions[["ID", "Topic_Short", "Session_Duration"]], on="ID", how="left"
)

# Mark as attended if total time >= threshold × session duration
per_session["Attended"] = (
    per_session["Total_Time_Min"] >=
    per_session["Session_Duration"] * ATTENDANCE_THRESHOLD_PCT
).astype(int)

# What % of the session did they attend (capped at 100%)
per_session["Session_Pct"] = (
    (per_session["Total_Time_Min"] / per_session["Session_Duration"]) * 100
).clip(upper=100).round(2)

print(f"  Attendance threshold : >= {ATTENDANCE_THRESHOLD_PCT*100:.0f}% of session duration")


# ════════════════════════════════════════════════════════════════
# STEP 5 — STUDENT-LEVEL SUMMARY
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 5 — Building Student Summary Table")
print("=" * 65)

summary = (
    per_session.groupby("Email")
    .agg(
        Name=("Name", "first"),
        Classes_Attended=("Attended", "sum"),
        Avg_Time_Per_Session=("Total_Time_Min", "mean"),
        Total_Join_Count=("Join_Count", "sum"),
        Avg_Session_Pct=("Session_Pct", "mean"),
        Sessions_Present=("Total_Time_Min", lambda x: (x > 0).sum()),
    )
    .reset_index()
)

summary["Total_Classes"]         = TOTAL_SESSIONS
summary["Attendance_Percentage"] = (
    (summary["Classes_Attended"] / TOTAL_SESSIONS) * 100
).round(2)

# Engagement Score — composite metric (0 to 100)
#   60% weight → overall attendance %
#   30% weight → how deeply they attended each session
#   10% weight → consistency (showed up at all, even briefly)
summary["Engagement_Score"] = (
    0.60 * summary["Attendance_Percentage"]
    + 0.30 * summary["Avg_Session_Pct"]
    + 0.10 * (summary["Sessions_Present"] / TOTAL_SESSIONS * 100)
).round(2)

# Rule-based certificate label
summary["Certificate_Status"] = summary["Attendance_Percentage"].apply(
    lambda x: "Issued" if x >= CERTIFICATE_THRESHOLD else "Not Issued"
)

issued = (summary["Certificate_Status"] == "Issued").sum()
print(f"  Total unique students : {len(summary):,}")
print(f"  Certificates Issued   : {issued:,}  ({issued/len(summary)*100:.1f}%)")
print(f"  Not Issued            : {len(summary)-issued:,}  ({(len(summary)-issued)/len(summary)*100:.1f}%)")


# ════════════════════════════════════════════════════════════════
# STEP 6 — MACHINE LEARNING
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 6 — Machine Learning Models")
print("=" * 65)

FEATURES = [
    "Classes_Attended",        # sessions they fully attended
    "Avg_Time_Per_Session",    # average minutes spent per session
    "Total_Join_Count",        # re-joins = engagement signal
    "Avg_Session_Pct",         # avg % of each session covered
    "Sessions_Present",        # sessions where they showed up at all
    "Engagement_Score",        # composite score
]

X = summary[FEATURES].fillna(0)
y = (summary["Certificate_Status"] == "Issued").astype(int)

print(f"\n  Label distribution → Issued: {y.sum()} | Not Issued: {len(y) - y.sum()}")

if y.nunique() < 2:
    print("\n  WARNING: Only one class label found in this dataset.")
    print("  This usually means the dataset covers too few sessions.")
    print("  The certificate threshold can't be crossed with limited data.")
    print("  Fix: Add more CSV files (more class days) to CSV_FILES above.")
    print("  Skipping ML training — attendance report will still be saved.\n")
    ml_ran = False

else:
    ml_ran = True

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    model_zoo = {
        "Random Forest      ": RandomForestClassifier(n_estimators=150, random_state=42),
        "Gradient Boosting  ": GradientBoostingClassifier(n_estimators=150, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    }

    results   = {}
    best_acc  = 0
    best_name = None
    best_model = None

    for name, model in model_zoo.items():
        is_lr = "Logistic" in name
        Xtr = X_train_sc if is_lr else X_train
        Xte = X_test_sc  if is_lr else X_test
        Xfull = scaler.transform(X) if is_lr else X

        model.fit(Xtr, y_train)
        preds  = model.predict(Xte)
        cv_acc = cross_val_score(model, Xfull, y, cv=cv, scoring="accuracy")
        acc    = accuracy_score(y_test, preds)

        results[name] = acc

        print(f"\n  [{name}]")
        print(f"    Test Accuracy : {acc*100:.2f}%")
        print(f"    CV  Accuracy  : {cv_acc.mean()*100:.2f}% ± {cv_acc.std()*100:.2f}%")
        print(classification_report(
            y_test, preds,
            target_names=["Not Issued", "Issued"],
            zero_division=0
        ))

        if acc > best_acc:
            best_acc   = acc
            best_name  = name
            best_model = model
            best_scaler_used = is_lr

    print(f"\n  Best Model : {best_name.strip()} — {best_acc*100:.2f}% accuracy")

    # Feature importance bar chart (tree models only)
    if hasattr(best_model, "feature_importances_"):
        print("\n  Feature Importances:")
        imp = pd.Series(
            best_model.feature_importances_, index=FEATURES
        ).sort_values(ascending=False)
        for feat, val in imp.items():
            bar = "=" * int(val * 50)
            print(f"    {feat:<28} {bar}  {val:.4f}")

    # Add ML prediction column to summary
    X_pred = scaler.transform(X) if best_scaler_used else X
    summary["ML_Predicted_Status"] = best_model.predict(X_pred)
    summary["ML_Predicted_Status"] = summary["ML_Predicted_Status"].map(
        {1: "Issued", 0: "Not Issued"}
    )


# ════════════════════════════════════════════════════════════════
# STEP 7 — FINAL OUTPUT TABLE
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 7 — Final Output")
print("=" * 65)

output_cols = [
    "Name", "Email", "Total_Classes", "Classes_Attended",
    "Attendance_Percentage", "Engagement_Score", "Certificate_Status"
]
if ml_ran:
    output_cols.append("ML_Predicted_Status")

output = (
    summary[output_cols]
    .sort_values("Attendance_Percentage", ascending=False)
    .reset_index(drop=True)
)
output.index     += 1
output.index.name = "Student_ID"

print(f"\n  Top 15 students by attendance:\n")
print(output.head(15).to_string())

# Save to CSV
output.to_csv(OUTPUT_CSV)
print(f"\n  Report saved → {OUTPUT_CSV}")

# ────────────────────────────────────────
# FINAL SUMMARY
# ────────────────────────────────────────
print("\n" + "=" * 65)
print("  SUMMARY")
print("=" * 65)
print(f"  Total students  : {len(output):,}")
print(f"  Total sessions  : {TOTAL_SESSIONS}")
print(f"  Cert threshold  : >= {CERTIFICATE_THRESHOLD}% attendance")
print(f"  Issued          : {(output['Certificate_Status'] == 'Issued').sum():,}")
print(f"  Not Issued      : {(output['Certificate_Status'] == 'Not Issued').sum():,}")
print(f"\n  Attendance % statistics:")
print(output["Attendance_Percentage"].describe().to_string())

  STEP 1 — Loading & Merging CSV Files
  Loaded: meetinglistdetails_2026_05_30_2026_05_30.csv
  Loaded: meetinglistdetails_2026_05_29_2026_05_29.csv

  Total rows loaded : 9,859
  Unique meeting IDs: 4

  STEP 2 — Session Information
  [2026-05-30] Data Analysis (Python)         — 1 min
  [2026-05-30] Machine Learning               — 122 min
  [2026-05-30] Cloud & DevOps                 — 28 min
  [2026-05-30] Embedded Systems               — 75 min

  Total classes (sessions) in dataset: 4

  STEP 3 — Cleaning Student Records
  Student rows (after filter) : 9,846
  Unique student emails       : 2,521

  STEP 4 — Computing Per-Session Attendance
  Attendance threshold : >= 50% of session duration

  STEP 5 — Building Student Summary Table
  Total unique students : 2,521
  Certificates Issued   : 0  (0.0%)
  Not Issued            : 2,521  (100.0%)

  STEP 6 — Machine Learning Models

  Label distribution → Issued: 0 | Not Issued: 2521

  This usually means the dataset covers too few ses